# StatArb LSTM Pipeline — Cross-Exchange Z-Score Forecast

**Train:** everything before **2026-07-25** (incl. Jul 22-24 from jul22-28 run)  
**Val:** chronological carve from train (early stopping only)  
**Test:** **Jul 25-28** (`snapshot_idx >= 3584`)

**Target:** `y_t = z_{t+1}` rolling z of cross-exchange `spread_bps` (W=300, min_periods=90)  
**Model:** 2-layer PyTorch LSTM + coin/pair embeddings  
**Trade filter:** `|pred| > 0.5` (same policy axis as LGBM campaigns)

Features are **LSTM-native sequences** from HF/local CEX tables — not the 68 LGBM columns.

Engineering prompt: `docs/prompts/lstm_zscore_engineering_prompt.md`  
Library: `statarb/lstm_zscore_lib.py`  
CLI runner: `statarb/run_lstm_zscore.py`

## 1. Imports & Config

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if (HERE / "lstm_zscore_lib.py").exists():
    sys.path.insert(0, str(HERE))
    OUTPUT_DIR = HERE / "outputs_lstm"
elif (HERE / "statarb" / "lstm_zscore_lib.py").exists():
    sys.path.insert(0, str(HERE / "statarb"))
    OUTPUT_DIR = HERE / "statarb" / "outputs_lstm"
else:
    raise RuntimeError("Cannot find lstm_zscore_lib.py — run from repo root or statarb/")

import lstm_zscore_lib as L

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_HF = True
LOCAL_DATA_ROOT = L.resolve_local_data_root()
HF_TOKEN = L.resolve_hf_token()
CFG = L.TrainConfig()
TRAIN_STRIDE = 5   # every Nth decision time (memory)
TEST_STRIDE = 2
L.set_seed(CFG.seed)

print("LOCAL_DATA_ROOT:", LOCAL_DATA_ROOT if LOCAL_DATA_ROOT else "(none — will use Hugging Face)")
print("HF auth:", "yes" if HF_TOKEN else "no")
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("Protocol:", {
    "HORIZON": L.HORIZON,
    "ZSCORE_WINDOW": L.ZSCORE_WINDOW,
    "MIN_PERIODS": L.MIN_PERIODS,
    "SEQ_LEN": CFG.seq_len,
    "ENTRY_TAU": CFG.entry_tau,
    "TRAIN_STRIDE": TRAIN_STRIDE,
    "TEST_STRIDE": TEST_STRIDE,
})

## 2. Data Loading
Same `WINDOWS` / Jul-25 `snapshot_idx` cut as `cex_gbm_new.ipynb`. Drops error rows on load.

In [ ]:
train_windows = [w for w in L.WINDOWS if w["role"] == "train"]
test_windows = [w for w in L.WINDOWS if w["role"] == "test"]

print("=" * 60)
print("Loading TRAIN pool (all data before Jul 25) …")
print("=" * 60)
train_raw = L.pool_windows(
    train_windows, local_root=LOCAL_DATA_ROOT, hf_token=HF_TOKEN, use_hf=USE_HF
)

print("\n" + "=" * 60)
print("Loading TEST pool (Jul 25-28) …")
print("=" * 60)
test_raw = L.pool_windows(
    test_windows, local_root=LOCAL_DATA_ROOT, hf_token=HF_TOKEN, use_hf=USE_HF
)

print("\nDone. Final table sizes:")
for table in L.SUBSETS:
    tr = len(train_raw.get(table, []))
    te = len(test_raw.get(table, []))
    print(f"  {table:20s}  train={tr:>10,}  test={te:>10,}")

assert len(train_raw["spread_matrix"]) > 0, "train spread_matrix empty"
assert len(test_raw["spread_matrix"]) > 0, "test spread_matrix empty"
print("\n[Phase 1 checkpoint] train/test pools loaded.")

## 3-8. Panel + LSTM-native sequence features

Build pairwise spread z-target, join pair-leg ticker/OB/trades, light cross-venue stats, then cut sequences of length `SEQ_LEN` **within** each `window_id`.

In [ ]:
import gc

print("Building train panel …")
train_panel = L.build_panel(train_raw)
del train_raw
gc.collect()
print("Building test panel …")
test_panel = L.build_panel(test_raw)
del test_raw
gc.collect()

train_feat = L.apply_row_transforms(L.panel_to_feature_frame(train_panel))
del train_panel
gc.collect()
test_feat = L.apply_row_transforms(L.panel_to_feature_frame(test_panel))
del test_panel
gc.collect()

print("train_feat", train_feat.shape, "test_feat", test_feat.shape)
print("Channels:", L.CHANNEL_NAMES)

tr_valid = train_feat.dropna(subset=["target", "zscore"])
null_rates = tr_valid[L.CHANNEL_NAMES].isna().mean().sort_values(ascending=False)
print("\nNull rates on train rows with valid target/z:")
print(null_rates.to_string())
print(f"\n% |target|>0.5: {(tr_valid['target'].abs() > 0.5).mean():.2%}")
print(f"target describe:\n{tr_valid['target'].describe()}")

In [ ]:
winsor_bounds = L.winsorize_fit(train_feat, L.WINSOR_CHANNELS)
train_feat = L.winsorize_apply(train_feat, winsor_bounds)
test_feat = L.winsorize_apply(test_feat, winsor_bounds)

train_bundle, coin_to_id, pair_to_id = L.build_sequences(
    train_feat, seq_len=CFG.seq_len, stride=TRAIN_STRIDE
)
test_bundle, _, _ = L.build_sequences(
    test_feat,
    seq_len=CFG.seq_len,
    coin_to_id=coin_to_id,
    pair_to_id=pair_to_id,
    stride=TEST_STRIDE,
)
del train_feat, test_feat
gc.collect()

print("train sequences", train_bundle.X.shape, "y", train_bundle.y.shape)
print("test sequences ", test_bundle.X.shape, "y", test_bundle.y.shape)
print("n_coins", len(coin_to_id), "n_pairs", len(pair_to_id))

assert train_bundle.meta["window_id"].nunique() >= 1
assert len(train_bundle.y) > 0 and len(test_bundle.y) > 0
print("\n[Phase 2 checkpoint] sequences ready.")

## 9-10. Train / val tensors + ZScoreLSTM

In [ ]:
train_mask, val_mask = L.chronological_val_split(train_bundle, val_fraction=CFG.val_fraction)
print(f"train n={train_mask.sum():,}  val n={val_mask.sum():,}  test n={len(test_bundle.y):,}")

scaler = L.fit_scaler(train_bundle.X[train_mask])
X_tr = L.transform_X(train_bundle.X[train_mask], scaler)
X_va = L.transform_X(train_bundle.X[val_mask], scaler)
X_te = L.transform_X(test_bundle.X, scaler)

y_tr = train_bundle.y[train_mask]
y_va = train_bundle.y[val_mask]
y_te = test_bundle.y
c_tr, p_tr = train_bundle.coin_id[train_mask], train_bundle.pair_id[train_mask]
c_va, p_va = train_bundle.coin_id[val_mask], train_bundle.pair_id[val_mask]
c_te, p_te = test_bundle.coin_id, test_bundle.pair_id
z_te = test_bundle.z_now

model = L.ZScoreLSTM(
    n_channels=len(L.CHANNEL_NAMES),
    n_coins=len(coin_to_id),
    n_pairs=len(pair_to_id),
    cfg=CFG,
)
print("device:", model.device)

history = model.fit(
    X_tr, y_tr, c_tr, p_tr,
    X_va, y_va, c_va, p_va,
    checkpoint_path=OUTPUT_DIR / "best_val.pt",
)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history["train_mse"], label="train_mse")
ax.plot(history["val_mse"], label="val_mse")
ax.set_xlabel("epoch")
ax.legend()
ax.set_title("LSTM training curves")
plt.show()
print("[Phase 3 checkpoint] training complete; best weights loaded.")

## 11. Evaluate (all + `|pred| > 0.5` vs naive `z_t`)

In [ ]:
pred_te = model.predict(X_te, c_te, p_te)
metrics = L.evaluate_model_and_naive(
    y_te, pred_te, z_te, tau=CFG.entry_tau, meta=test_bundle.meta
)

print("=== HEADLINE (filtered |pred|>tau) — primary vs LGBM ===")
print("LSTM :", metrics["lstm"]["headline_filtered"])
print("Naive:", metrics["naive_zt"]["headline_filtered"])

def _show(name, block):
    print(f"\n=== {name} ===")
    for slice_name, m in block.items():
        if slice_name == "notes":
            continue
        print(f"  {slice_name}: {m}")

_show("LSTM", metrics["lstm"])
_show("Naive z_t", metrics["naive_zt"])

import json as _json
(OUTPUT_DIR / "metrics.json").write_text(_json.dumps(metrics, indent=2), encoding="utf-8")
print("\n[Phase 4 checkpoint] metrics written to", OUTPUT_DIR / "metrics.json")


## 12-13. Export artifacts + METRICS.md

In [ ]:
L.export_artifacts(
    output_dir=OUTPUT_DIR,
    model=model,
    scaler=scaler,
    coin_to_id=coin_to_id,
    pair_to_id=pair_to_id,
    channel_names=L.CHANNEL_NAMES,
    cfg=CFG,
    metrics=metrics,
    winsor_bounds=winsor_bounds,
)
print("\n[Phase 5 checkpoint] export complete.")
print(list(OUTPUT_DIR.glob("*")))